# House Price Prediction: Regression Project
**Research question:** Can machine learning accurately predict house selling prices from property characteristics, and which regression algorithm performs best?

**Target:** `median_house_value` (continuous, USD)  |  **Metrics:** MAE, MSE, RMSE, R²

> **Dataset note.** `housing.csv` is the California Housing dataset. Each row is a census *block group* (a neighbourhood), so the target is a neighbourhood's median house value and the features are neighbourhood-level (income, average rooms, location, ocean proximity ...), not single-house attributes such as garage size or condition.

## Project workflow

In [ ]:
import numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt
import os, joblib, warnings, time
from pathlib import Path
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from IPython.display import Markdown, display
from sklearn.model_selection import train_test_split, cross_validate, KFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
FAST_MODE = False   # set True for a quick, lighter hyper-parameter search (fewer iterations)

In [ ]:
stages = [
    ("1. Data collection",     "Load housing.csv",         "prep"),
    ("2. Data cleaning",       "Fix gaps, check caps",     "prep"),
    ("3. EDA",                 "Explore patterns",         "prep"),
    ("4. Feature engineering", "Ratios, city distances",   "prep"),
    ("5. Train / test split",  "80% train, 20% test",      "model"),
    ("6. Model training",      "Fit 5 algorithms",         "model"),
    ("7. Model comparison",    "Cross-validate, tune",     "model"),
    ("8. Evaluation",          "MAE, MSE, RMSE, R\u00b2", "model"),
    ("9. Price prediction",    "Estimate new value",       "model"),
]
colors = {"prep": ("#E1F5EE", "#0F6E56", "#085041", "#0F6E56"),
          "model": ("#EEEDFE", "#534AB7", "#3C3489", "#534AB7")}
xs = [0, 1, 2, 2, 1, 0, 0, 1, 2]       # snake layout: L->R, down, R->L, down, L->R
ys = [2, 2, 2, 1, 1, 1, 0, 0, 0]
GX, GY, W, H = 3.2, 1.6, 2.6, 1.0

fig, ax = plt.subplots(figsize=(11, 6.2)); ax.axis("off")
ax.set_xlim(-0.4, 2 * GX + W + 0.4); ax.set_ylim(-0.7, 2 * GY + H + 0.3); ax.set_aspect("equal")
pos = []
for (title, sub, kind), cx, cy in zip(stages, xs, ys):
    x, y = cx * GX, cy * GY
    fill, edge, t_col, s_col = colors[kind]
    ax.add_patch(FancyBboxPatch((x, y), W, H, boxstyle="round,pad=0,rounding_size=0.08", fc=fill, ec=edge, lw=1.2))
    ax.text(x + W / 2, y + H * 0.62, title, ha="center", va="center", fontsize=11, fontweight="bold", color=t_col)
    ax.text(x + W / 2, y + H * 0.28, sub, ha="center", va="center", fontsize=9.5, color=s_col)
    pos.append((x, y))

def arrow(p, q):
    ax.add_patch(FancyArrowPatch(p, q, arrowstyle="-|>", mutation_scale=16, lw=1.3, color="#555"))

g = 0.08
for i in range(8):
    (x1, y1), (x2, y2) = pos[i], pos[i + 1]
    if y1 == y2 and x2 > x1:   arrow((x1 + W + g, y1 + H / 2), (x2 - g, y2 + H / 2))
    elif y1 == y2:             arrow((x1 - g, y1 + H / 2), (x2 + W + g, y2 + H / 2))
    else:                      arrow((x1 + W / 2, y1 - g), (x2 + W / 2, y2 + H + g))

ax.add_patch(FancyBboxPatch((0.0, -0.55), 0.28, 0.2, boxstyle="round,pad=0,rounding_size=0.03", fc="#E1F5EE", ec="#0F6E56"))
ax.text(0.4, -0.45, "Data preparation (stages 1-4)", va="center", fontsize=10)
ax.add_patch(FancyBboxPatch((3.6, -0.55), 0.28, 0.2, boxstyle="round,pad=0,rounding_size=0.03", fc="#EEEDFE", ec="#534AB7"))
ax.text(4.0, -0.45, "Modelling and results (stages 5-9)", va="center", fontsize=10)
ax.set_title("House price prediction: project workflow", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.savefig("workflow.png", dpi=150, bbox_inches="tight"); plt.show()

| Stage | What happens |
|---|---|
| 1. Data collection | Load the CSV, check shape, types and a sample of rows. |
| 2. Data cleaning | Find missing values, duplicates and capped values; decide how to treat each. |
| 3. EDA | Plot the target, feature distributions, correlations, geography and key relationships. |
| 4. Feature engineering | Create per-household ratios and distances to major cities; encode categories. |
| 5. Train / test split | Hold out 20% of the data (stratified by income band) and build preprocessing pipelines. |
| 6. Model training | Fit Linear Regression, Decision Tree, Random Forest, Gradient Boosting and XGBoost. |
| 7. Model comparison | 5-fold cross-validation, then hyper-parameter tuning of the strongest tree models. |
| 8. Evaluation | Score all models on the held-out test set with MAE, MSE, RMSE and R²; inspect errors and feature importance. |
| 9. Price prediction | Wrap the best model in a function that estimates the price for a new area. |

# Stage 1: Data collection

In [ ]:
# Find housing.csv: next to the notebook, in Downloads, or (on Colab) via an upload dialog.
candidates = [Path("housing.csv"), Path("/content/housing.csv"),
              Path.home() / "Downloads" / "housing.csv", Path("/mnt/user-data/uploads/housing.csv")]
DATA_PATH = next((p for p in candidates if p.exists()), None)

if DATA_PATH is None:
    try:
        from google.colab import files
        print("housing.csv not found. Please upload the file.")
        uploaded = files.upload()
        for filename in uploaded.keys():
            if filename != "housing.csv":
                os.rename(filename, "housing.csv")
        DATA_PATH = Path("housing.csv")
    except ImportError:
        raise FileNotFoundError("housing.csv not found. Put it next to this notebook or set DATA_PATH manually.")

print("Loading:", DATA_PATH)
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

In [ ]:
df.info()
df.describe().T

# Stage 2: Data cleaning

In [ ]:
print("Missing values:\n", df.isna().sum()[df.isna().sum() > 0])
print("\nDuplicate rows:", df.duplicated().sum())
print("\nocean_proximity counts:\n", df["ocean_proximity"].value_counts())
print("\nRows at the price cap (500,001):", (df["median_house_value"] >= 500001).sum())
print("Rows at the age cap (52):", (df["housing_median_age"] >= 52).sum())

**Findings and decisions**
- `total_bedrooms` has 207 missing values (1%). They are imputed with the **median inside the modelling pipeline**, fitted on training data only, so nothing leaks from the test set.
- `median_house_value` is **censored at \$500,001** (965 rows at exactly that value) and `housing_median_age` at 52. Models cannot learn true values above the cap, so errors will be larger for expensive areas. These rows are **kept** so the evaluation reflects the real data, and this is flagged as a limitation.
- `ISLAND` has only 5 rows. It is kept; one-hot encoding handles it and unseen categories are ignored safely.

# Stage 3: Exploratory data analysis (EDA)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(df["median_house_value"], bins=60, ax=axes[0], color="#2a6f97")
axes[0].set_title("Target: median house value (note the spike at the cap)")
sns.histplot(np.log1p(df["median_house_value"]), bins=60, ax=axes[1], color="#61a5c2")
axes[1].set_title("log(1 + median house value)")
plt.tight_layout(); plt.show()

In [ ]:
num_cols = df.select_dtypes("number").columns
df[num_cols].hist(bins=50, figsize=(14, 9), color="#468faf")
plt.suptitle("Feature distributions (counts are heavily right-skewed)", y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(df[num_cols].corr(), annot=True, fmt=".2f", cmap="RdBu_r", center=0)
plt.title("Correlation matrix"); plt.show()
print(df[num_cols].corr()["median_house_value"].sort_values(ascending=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sc = axes[0].scatter(df["longitude"], df["latitude"], c=df["median_house_value"],
                     s=df["population"]/200, alpha=0.4, cmap="viridis")
axes[0].set_title("Geography: colour = price, size = population")
axes[0].set_xlabel("longitude"); axes[0].set_ylabel("latitude")
plt.colorbar(sc, ax=axes[0], label="median house value")
sns.boxplot(data=df, x="ocean_proximity", y="median_house_value", ax=axes[1],
            order=["INLAND", "<1H OCEAN", "NEAR OCEAN", "NEAR BAY", "ISLAND"])
axes[1].set_title("Price by ocean proximity")
plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df.sample(5000, random_state=1), x="median_income", y="median_house_value", alpha=0.3)
plt.title("Income vs price (horizontal band at the $500k cap)"); plt.show()

**EDA takeaways**
- `median_income` is by far the strongest linear correlate of price.
- Location matters a lot: coastal and Bay Area / LA blocks are far more expensive than inland ones. Latitude and longitude carry non-linear structure that tree models should exploit better than a linear model.
- Count features (`total_rooms`, `population`, `households`) are skewed and strongly correlated with each other because they mostly measure block-group size. Per-household ratios will be more informative.

# Stage 4: Feature engineering

In [ ]:
# Distance to major California population centres (haversine, km)
CITIES = {"la": (34.05, -118.24), "sf": (37.77, -122.42), "sd": (32.72, -117.16),
          "sj": (37.34, -121.89), "sac": (38.58, -121.49)}

def haversine_km(lat1, lon1, lat2, lon2):
    p = np.pi / 180
    a = (np.sin((lat2 - lat1) * p / 2) ** 2
         + np.cos(lat1 * p) * np.cos(lat2 * p) * np.sin((lon2 - lon1) * p / 2) ** 2)
    return 12742 * np.arcsin(np.sqrt(a))

def add_features(d):
    d = d.copy()
    d["rooms_per_household"]  = d["total_rooms"] / d["households"]
    d["bedrooms_per_room"]    = d["total_bedrooms"] / d["total_rooms"]
    d["people_per_household"] = d["population"] / d["households"]
    dists = np.column_stack([haversine_km(d["latitude"], d["longitude"], la, lo) for la, lo in CITIES.values()])
    d["dist_nearest_city_km"] = dists.min(axis=1)
    d["dist_la_km"] = dists[:, 0]
    d["dist_sf_km"] = dists[:, 1]
    return d

df_fe = add_features(df)
df_fe[["rooms_per_household", "bedrooms_per_room", "people_per_household", "dist_nearest_city_km"]].describe().T

In [ ]:
print(df_fe.drop(columns="ocean_proximity").corr()["median_house_value"].sort_values(ascending=False).round(3))

These are row-wise ratios and distances to fixed coordinates, so computing them before the split uses no information from other rows and causes no leakage. The scaling, imputation and encoding steps that *do* learn from data stay inside the pipeline.

# Stage 5: Train / test split

In [ ]:
TARGET = "median_house_value"
X = df_fe.drop(columns=TARGET)
y = df_fe[TARGET]

# Stratify on income bands so train and test have the same income mix
income_band = pd.cut(df_fe["median_income"], [0, 1.5, 3, 4.5, 6, np.inf], labels=[1, 2, 3, 4, 5])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=income_band)
print(X_train.shape, X_test.shape)

## Stage 5 (continued): Preprocessing pipelines

In [ ]:
cat_cols = ["ocean_proximity"]
num_cols_model = [c for c in X.columns if c not in cat_cols]
skewed = ["total_rooms", "total_bedrooms", "population", "households",
          "rooms_per_household", "people_per_household", "dist_nearest_city_km"]
other_num = [c for c in num_cols_model if c not in skewed]

# Linear model: impute -> log1p the skewed columns -> scale; one-hot for the category
linear_prep = ColumnTransformer([
    ("skewed", Pipeline([("imp", SimpleImputer(strategy="median")),
                         ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
                         ("sc", StandardScaler())]), skewed),
    ("other", Pipeline([("imp", SimpleImputer(strategy="median")),
                        ("sc", StandardScaler())]), other_num),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])
# Tree models: no scaling needed; just impute and one-hot encode
tree_prep = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num_cols_model),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])

# Stage 6: Model training

In [ ]:
models = {
    "Linear Regression": Pipeline([("prep", linear_prep), ("model", LinearRegression())]),
    "Linear Regression (log target)": Pipeline([("prep", linear_prep),
        ("model", TransformedTargetRegressor(LinearRegression(), func=np.log1p, inverse_func=np.expm1))]),
    "Decision Tree": Pipeline([("prep", tree_prep),
        ("model", DecisionTreeRegressor(max_depth=10, min_samples_leaf=5, random_state=RANDOM_STATE))]),
    "Random Forest": Pipeline([("prep", tree_prep),
        ("model", RandomForestRegressor(n_estimators=150, max_features=0.5, n_jobs=-1, random_state=RANDOM_STATE))]),
    "Gradient Boosting": Pipeline([("prep", tree_prep),
        ("model", GradientBoostingRegressor(n_estimators=200, learning_rate=0.1, max_depth=4,
                                            subsample=0.8, random_state=RANDOM_STATE))]),
    "XGBoost": Pipeline([("prep", tree_prep),
        ("model", XGBRegressor(n_estimators=400, learning_rate=0.08, max_depth=6, subsample=0.8,
                               colsample_bytree=0.8, n_jobs=-1, random_state=RANDOM_STATE))]),
}

# Stage 7: Model comparison (5-fold cross-validation)

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {"mae": "neg_mean_absolute_error", "rmse": "neg_root_mean_squared_error", "r2": "r2"}
cv_rows = {}
for name, m in models.items():
    s = cross_validate(m, X_train, y_train, cv=kf, scoring=scoring, n_jobs=1)
    cv_rows[name] = {"CV MAE": -s["test_mae"].mean(), "CV RMSE": -s["test_rmse"].mean(),
                     "CV RMSE std": s["test_rmse"].std(), "CV R2": s["test_r2"].mean()}
    print("done:", name)
cv_results = pd.DataFrame(cv_rows).T.sort_values("CV RMSE")
cv_results.index.name = "Model"
cv_results.round(3)

## Stage 7 (continued): Hyperparameter tuning of the top tree-based models

In [ ]:
search_spaces = {
    "XGBoost": {
        "model__n_estimators": [300, 500, 800],
        "model__learning_rate": [0.03, 0.05, 0.08, 0.1],
        "model__max_depth": [4, 6, 8],
        "model__min_child_weight": [1, 5, 10],
        "model__subsample": [0.7, 0.85, 1.0],
        "model__colsample_bytree": [0.6, 0.8, 1.0],
    },
    "Random Forest": {
        "model__n_estimators": [150, 250],
        "model__max_depth": [None, 25],
        "model__min_samples_leaf": [1, 2, 3],
        "model__max_features": [0.4, 0.6],
    },
}
n_iters = {"XGBoost": 3 if FAST_MODE else 15, "Random Forest": 3 if FAST_MODE else 8}
tuned = {}
for name, space in search_spaces.items():
    rs = RandomizedSearchCV(models[name], space, n_iter=n_iters[name], cv=3,
                            scoring="neg_root_mean_squared_error",
                            random_state=RANDOM_STATE, n_jobs=1, refit=True)
    rs.fit(X_train, y_train)
    tuned[name + " (tuned)"] = rs.best_estimator_
    print(f"{name}: best CV RMSE = {-rs.best_score_:,.0f}")
    print("   ", {k.replace("model__", ""): v for k, v in rs.best_params_.items()})

# Stage 8: Final evaluation on the held-out test set

In [ ]:
def metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {"MAE": mean_absolute_error(y_true, y_pred), "MSE": mse,
            "RMSE": np.sqrt(mse), "R2": r2_score(y_true, y_pred)}

final_models = dict(models); final_models.update(tuned)
test_rows, fitted = {}, {}
for name, m in final_models.items():
    m.fit(X_train, y_train)          # refit on the full training set (tuned or not)
    fitted[name] = m
    test_rows[name] = metrics(y_test, m.predict(X_test))
test_results = pd.DataFrame(test_rows).T.sort_values("RMSE")
test_results.round(3)

In [ ]:
best_name = test_results.index[0]
print("Best model on the test set:", best_name)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
order = test_results.index
sns.barplot(x=test_results.loc[order, "RMSE"], y=order, ax=axes[0], color="#2a6f97")
axes[0].set_title("Test RMSE ($) - lower is better")
sns.barplot(x=test_results.loc[order, "R2"], y=order, ax=axes[1], color="#61a5c2")
axes[1].set_title("Test R2 - higher is better"); axes[1].set_xlim(0.5, 1)
plt.tight_layout(); plt.show()

In [ ]:
best = fitted[best_name]
pred = best.predict(X_test)
resid = y_test - pred
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
axes[0].scatter(y_test, pred, alpha=0.25, s=8)
lim = [0, 520000]; axes[0].plot(lim, lim, "r--"); axes[0].set_xlim(lim); axes[0].set_ylim(lim)
axes[0].set_xlabel("Actual"); axes[0].set_ylabel("Predicted"); axes[0].set_title(f"Predicted vs actual ({best_name})")
axes[1].scatter(pred, resid, alpha=0.25, s=8); axes[1].axhline(0, color="r", ls="--")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Residual"); axes[1].set_title("Residuals vs predicted")
sns.histplot(resid, bins=60, ax=axes[2], color="#468faf"); axes[2].set_title("Residual distribution")
plt.tight_layout(); plt.show()

In [ ]:
# Where does the model do worst? Compare error on capped vs uncapped rows
capped = y_test >= 500001
print(f"RMSE on rows below the cap : {np.sqrt(mean_squared_error(y_test[~capped], pred[~capped])):,.0f}")
print(f"RMSE on capped rows (n={capped.sum()}) : {np.sqrt(mean_squared_error(y_test[capped], pred[capped])):,.0f}")
print(f"MAE as % of mean price      : {mean_absolute_error(y_test, pred) / y_test.mean():.1%}")

## Stage 8: Which features drive price?

In [ ]:
pi = permutation_importance(best, X_test, y_test, n_repeats=5, random_state=RANDOM_STATE,
                            scoring="neg_root_mean_squared_error", n_jobs=1)
imp = pd.Series(pi.importances_mean, index=X_test.columns).sort_values()
imp.plot.barh(figsize=(8, 6), color="#2a6f97")
plt.xlabel("Increase in RMSE ($) when the feature is shuffled")
plt.title(f"Permutation importance ({best_name})"); plt.show()

# Stage 9: Price prediction for a new area

In [ ]:
joblib.dump(best, "best_house_price_model.joblib")

def predict_price(longitude, latitude, housing_median_age, total_rooms, total_bedrooms,
                  population, households, median_income, ocean_proximity, model=best):
    row = pd.DataFrame([dict(longitude=longitude, latitude=latitude, housing_median_age=housing_median_age,
                             total_rooms=total_rooms, total_bedrooms=total_bedrooms, population=population,
                             households=households, median_income=median_income,
                             ocean_proximity=ocean_proximity)])
    return float(model.predict(add_features(row)[X_train.columns])[0])

# Example: a neighbourhood near the SF Bay, mid-high income, fairly old housing
est = predict_price(-122.25, 37.85, 30, 2500, 500, 1200, 450, 5.5, "NEAR BAY")
print(f"Estimated median house value: ${est:,.0f}")

# Conclusions

In [ ]:
t = test_results
b = t.iloc[0]; lr = t.loc["Linear Regression"]
xgb_plain = t.loc["XGBoost"]
tbl = "| Model | MAE (\\$) | RMSE (\\$) | R\u00b2 |\n|---|---|---|---|\n" + "\n".join(
    f"| {n} | {r.MAE:,.0f} | {r.RMSE:,.0f} | {r.R2:.3f} |" for n, r in t.iterrows())
display(Markdown(f"**Results (held-out test set, {len(X_test):,} block groups)**\n\n{tbl}"))
display(Markdown(
    f"**Answer to the research question.** Yes, to a useful degree. The best model ({best_name}) explains about "
    f"{b.R2:.0%} of the variation in median house value and is off by about \\${b.MAE/1000:,.0f}k on average (MAE), "
    f"roughly {b.MAE/y_test.mean():.0%} of the mean price. Ensemble tree methods clearly beat the linear baseline "
    f"(R\u00b2 {b.R2:.2f} vs {lr.R2:.2f}) because price depends non-linearly on location and income. "
    f"Tuning gave a small gain ({1 - b.RMSE/xgb_plain.RMSE:.1%} lower RMSE than untuned XGBoost)."))

**Limitations**
- Rows are neighbourhoods, not individual houses, so predictions are block-level medians.
- Prices are capped at \$500,001, so the model cannot predict true values above the cap and errors are largest there.
- The data is a historical (1990 census) snapshot of California.
- No house-level attributes (bathrooms, garage, condition, renovation) exist in this file.

**Future work:** SHAP explanations, LightGBM / CatBoost, a house-level dataset (e.g. Ames, Iowa), and a small web app around `predict_price()`.